In [1]:
!pip install -U scikeras scikit-learn
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 58.8 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [2]:

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [3]:
data=pd.read_csv("/content/Churn_Modelling.csv")
data.head()
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)

In [4]:
#encode categorical variable
label_enc_gender=LabelEncoder()
data['Gender']=label_enc_gender.fit_transform(data['Gender'])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [5]:
#onehot encode geography col
from sklearn.preprocessing import OneHotEncoder
onehot_enc_geo=OneHotEncoder()
geo_enc=onehot_enc_geo.fit_transform(data[['Geography']])
geo_enc

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [6]:
onehot_enc_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [7]:
geo_enc_df=pd.DataFrame(geo_enc.toarray(),columns=onehot_enc_geo.get_feature_names_out(['Geography']))
geo_enc_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [8]:
data=pd.concat([data.drop('Geography',axis=1),geo_enc_df],axis=1)


In [9]:
##save the encoders and scaler
with open('label_enc_gender.pkl','wb') as file:
  pickle.dump(label_enc_gender,file)
with open('onehot_enc_geo.pkl','wb') as file:
  pickle.dump(onehot_enc_geo,file)


In [10]:
## dividing dataset
x=data.drop('Exited',axis=1)
y=data['Exited']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.fit_transform(x_test)

In [11]:
#define a func. to create the model and try diff parameters(kearClassifier)
def create_model(neurons=32,layers=1):
  model=Sequential()
  model.add(Dense(neurons,activation='relu',input_shape=(x_train.shape[1],)))
  for _ in range(layers-1):
    model.add(Dense(neurons,activation='relu'))
  model.add(Dense(1,activation='sigmoid'))
  model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [12]:
##create a keras classifier
model=KerasClassifier(model=create_model,verbose=0)

In [13]:
#defining the grid search parameters
param_grid={
    'model__neurons':[16,32,64,128],
    'model__layers':[1,2],
    'batch_size':[10,20],
    'epochs':[20]
}

In [15]:
early_stop = EarlyStopping(
    monitor='loss',
    patience=5,
    restore_best_weights=True
)
#perform grid search
grid= GridSearchCV(estimator=model,param_grid=param_grid,n_jobs=-1,cv=3,scoring='accuracy',verbose=2)
grid_result= grid.fit(x_train,y_train,callbacks=[early_stop])
#print the best paramters
print(f"Best: {grid_result.best_score_} using {grid_result.best_params_}")

Fitting 3 folds for each of 16 candidates, totalling 48 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best: 0.8568743704486302 using {'batch_size': 20, 'epochs': 20, 'model__layers': 1, 'model__neurons': 64}
